# Data Loading & Lyrics Cleaning

### Set Seed and Locate the Data

In [1]:
from pathlib import Path

import pandas as pd

seed = 42

candidate_paths = [
    Path("music4all-en"),
    Path("../music4all-en"),
]
data_dir = None
for path in candidate_paths:
    if path.exists():
        data_dir = path
        break

if data_dir is None:
    raise FileNotFoundError("Could not find music4all-en. Unzip the subset to the repo root.")

lyrics_dir = data_dir / "lyrics"

print("Data dir:", data_dir)
print("Lyrics dir:", lyrics_dir)

Data dir: music4all-en
Lyrics dir: music4all-en\lyrics


### Load the Five Metadata Tables

In [2]:
table_files = {
    "information": "id_information.csv",
    "lang": "id_lang.csv",
    "tags": "id_tags.csv",
    "genres": "id_genres.csv",
    "metadata": "id_metadata.csv",
}

tables = {}
for name in table_files:
    filename = table_files[name]
    tables[name] = pd.read_csv(data_dir / filename, sep="\t", dtype={"id": str})
    print(filename, "shape:", tables[name].shape)

tables["information"].head()

id_information.csv shape: (84103, 4)
id_lang.csv shape: (84103, 2)


id_tags.csv shape: (84103, 2)
id_genres.csv shape: (84103, 2)


id_metadata.csv shape: (84103, 11)


,id,artist,song,album_name
0,0009fFIM1eYThaPg,Cheryl,Rain on Me,3 Words
1,0010xmHR6UICBOYT,Oddisee,After Thoughts,The Beauty in All
2,002Jyd0vN4HyCpqL,Blue Öyster Cult,ME 262,Secret Treaties
3,006TYKNjNxWjfKjy,Rhapsody,Flames of Revenge,Legendary Years (Re-Recorded)
4,007LIJOPQ4Sb98qV,The Chameleons,Nostalgia,What Does Anything Mean? Basically (2009 Remas...


### Reconcile Row Counts
- All five tables should carry the same ids
- Every id should have one lyrics file

In [3]:
lyrics_ids = set()
for path in lyrics_dir.glob("*.txt"):
    lyrics_ids.add(path.stem)

id_sets = {}
for name in tables:
    df = tables[name]
    id_sets[name] = set(df["id"])
    print(name, "rows:", len(df), "unique ids:", df["id"].nunique(),
          "duplicated:", df["id"].duplicated().sum())

reference = id_sets["information"]
same_ids = True
for name in id_sets:
    if id_sets[name] != reference:
        same_ids = False

print("Lyrics files:", len(lyrics_ids))
print("Same id set across tables:", same_ids)
print("Ids without lyrics:", len(reference - lyrics_ids))
print("Lyrics without ids:", len(lyrics_ids - reference))

information rows: 84103 unique ids: 84103 duplicated: 0
lang rows: 84103 unique ids: 84103 duplicated: 0
tags rows: 84103 unique ids: 84103 duplicated: 0


genres rows: 84103 unique ids: 84103 duplicated: 0
metadata rows: 84103 unique ids: 84103 duplicated: 0
Lyrics files: 84103
Same id set across tables: True
Ids without lyrics: 0
Lyrics without ids: 0


### Check Required Columns
- artist for the artist-disjoint split
- six Spotify descriptors for the audio comparison
- duration_ms is present, so words per minute is possible

In [4]:
required_columns = {
    "information": ["artist", "song"],
    "lang": ["lang"],
    "tags": ["tags"],
    "genres": ["genres"],
    "metadata": ["danceability", "energy", "key", "mode", "valence", "tempo", "duration_ms"],
}

for name in required_columns:
    missing = []
    for column in required_columns[name]:
        if column not in tables[name].columns:
            missing.append(column)
    print(name, "missing columns:", missing)

print("Languages:", tables["lang"]["lang"].value_counts().to_dict())
print("Unique artists:", tables["information"]["artist"].nunique())
print("Metadata columns:", list(tables["metadata"].columns))

information missing columns: []
lang missing columns: []
tags missing columns: []
genres missing columns: []
metadata missing columns: []
Languages: {'en': 84103}
Unique artists: 11877
Metadata columns: ['id', 'spotify_id', 'popularity', 'release', 'danceability', 'energy', 'key', 'mode', 'valence', 'tempo', 'duration_ms']


### Lyrics Cleaning
- clean_lines keeps the line breaks for the line statistics later
- flatten is the lowercased single line version for the vectorisers
- Blank lines and section markers like [Chorus] are dropped

In [5]:
import re

marker_pattern = re.compile(r"^\s*\[[^\]]{1,30}\]\s*$")
def clean_lines(text):
    kept = []
    for line in text.replace("\r\n", "\n").replace("\r", "\n").split("\n"):
        line = line.rstrip()
        if not line.strip():
            continue
        if marker_pattern.match(line):
            continue
        kept.append(line)
    return "\n".join(kept)
def flatten(text):
    return re.sub(r"\s+", " ", clean_lines(text).lower()).strip()

sample_id = tables["information"]["id"].iloc[0]
sample_path = lyrics_dir / (sample_id + ".txt")
sample_text = sample_path.read_text(encoding="utf-8", errors="replace")

print("Raw characters:", len(sample_text))
print("Cleaned characters:", len(clean_lines(sample_text)))
print(clean_lines(sample_text)[:200])

Raw characters: 2648
Cleaned characters: 2631
It's a sunny day, so I got nowhere to hide
Not a cloud in the sky
So I'm pretending, I've got something in my eyes
Just so you won't see me cry
So I wait for the storm, ooh
To hide all my tears but it


### Build the Lyrics Table

In [6]:
import hashlib

records = []
for track_id in tables["information"]["id"]:
    path = lyrics_dir / (track_id + ".txt")
    raw = path.read_text(encoding="utf-8", errors="replace")
    structured = clean_lines(raw)
    flat = flatten(raw)
    n_lines = 0
    if structured != "":
        n_lines = structured.count("\n") + 1
    records.append({
        "id": track_id,
        "n_chars_raw": len(raw),
        "n_chars_clean": len(structured),
        "n_lines": n_lines,
        "n_words": len(flat.split()),
        "fingerprint": hashlib.sha1(flat.encode("utf-8")).hexdigest(),
    })

lyrics = pd.DataFrame(records)
lyrics = lyrics.merge(tables["information"][["id", "artist", "song"]], on="id")

print("Shape:", lyrics.shape)
print("Empty after cleaning:", (lyrics["n_words"] == 0).sum())
lyrics.head()

Shape: (84103, 8)
Empty after cleaning: 0


,id,n_chars_raw,n_chars_clean,n_lines,n_words,fingerprint,artist,song
0,0009fFIM1eYThaPg,2648,2631,75,576,30e58de4afb173a8f2e6e6d69524a0c7f4be5ff2,Cheryl,Rain on Me
1,0010xmHR6UICBOYT,1663,1663,33,300,1e45bda574851c4270ef4a678d331e1136aeb601,Oddisee,After Thoughts
2,002Jyd0vN4HyCpqL,1545,1537,40,287,350fe60aecb6d89214c55fa72caedf30bcc747c3,Blue Öyster Cult,ME 262
3,006TYKNjNxWjfKjy,816,811,24,158,4ce1ee5063104585c378ab4b483ceea67ad99e21,Rhapsody,Flames of Revenge
4,007LIJOPQ4Sb98qV,934,929,32,153,d3b831a5605e88080bd21bcaaf2d675d7f1e3c76,The Chameleons,Nostalgia


### Drop Exact Duplicates
- Same cleaned lyrics collapsed to the first id
- Some groups span different artists, which an artist-disjoint split would not catch

In [7]:
group_sizes = lyrics["fingerprint"].value_counts()
duplicate_hashes = group_sizes[group_sizes > 1].index

lyrics["is_duplicate"] = lyrics.duplicated(subset="fingerprint", keep="first")
clean = lyrics[~lyrics["is_duplicate"]].copy()

in_groups = lyrics[lyrics["fingerprint"].isin(duplicate_hashes)]
artists_per_group = in_groups.groupby("fingerprint")["artist"].nunique()
multi_artist = artists_per_group[artists_per_group > 1]

print("Duplicate groups:", len(duplicate_hashes))
print("Tracks dropped:", lyrics["is_duplicate"].sum())
print("Tracks kept:", len(clean))
print("Groups with more than one artist:", len(multi_artist))

largest = lyrics[lyrics["fingerprint"] == duplicate_hashes[0]]
print("Largest group:", len(largest), "tracks and", largest["artist"].nunique(), "artists")
largest[["artist", "song"]].head(8)

Duplicate groups: 1122
Tracks dropped: 1293
Tracks kept: 82810
Groups with more than one artist: 149
Largest group: 47 tracks and 34 artists


,artist,song
1633,f(x),12시 25분 Wish List
3506,Incubus,11 am
3790,Martin Jensen,16 Steps
4490,Yungblud,11 Minutes (with Halsey feat. Travis Barker)
4997,The Greeting Committee,17 - Single Version
9161,Danger,19:00
11414,Danger,11:02
13003,Wanna One,11


### Numeric Titles Got the Wrong Lyrics
- Tracks in duplicate groups start with a digit far more often than the rest
- The scraper that built the subset looks like it failed on numeric titles

In [8]:
starts_with_digit = lyrics["song"].astype(str).str.match(r"^\s*\d")

print("All tracks starting with a digit:", round(starts_with_digit.mean(), 3))
print("In duplicate groups:",
      round(starts_with_digit[lyrics["fingerprint"].isin(duplicate_hashes)].mean(), 3))
print("In the largest group:",
      round(starts_with_digit[lyrics["fingerprint"] == duplicate_hashes[0]].mean(), 3))

All tracks starting with a digit: 0.01
In duplicate groups: 0.062
In the largest group: 0.915


### Length After Cleaning

In [9]:
print("Tracks under 20 words:", (clean["n_words"] < 20).sum())
print("Characters removed by cleaning:",
      round(1 - clean["n_chars_clean"].sum() / clean["n_chars_raw"].sum(), 3))

clean[["n_lines", "n_words"]].describe()

Tracks under 20 words: 43
Characters removed by cleaning: 0.005


,n_lines,n_words
count,82810.000000,82810.000000
mean,39.594228,242.916061
std,19.242901,126.686406
min,1.000000,1.000000
25%,26.000000,157.000000
50%,37.000000,223.000000
75%,49.000000,302.000000
max,971.000000,4773.000000


### Save

In [10]:
out_dir = Path("data/processed")
out_dir.mkdir(parents=True, exist_ok=True)

out_columns = ["id", "artist", "song", "n_lines", "n_words", "fingerprint"]
out_path = out_dir / "lyrics_clean.csv"
clean[out_columns].to_csv(out_path, index=False)

print("Wrote:", out_path)
print("Rows:", len(clean))

Wrote: data\processed\lyrics_clean.csv
Rows: 82810


### Normalise the Tags
- Lowercase, trim, collapse repeated spaces
- Merge a small alias list so the same tag does not appear twice

In [11]:
tag_aliases = {
    "hiphop": "hip-hop",
    "hip hop": "hip-hop",
    "r&b": "rnb",
    "r and b": "rnb",
    "rock and roll": "rock n roll",
    "rock & roll": "rock n roll",
    "rock  roll": "rock n roll",
    "singer songwriter": "singer-songwriter",
    "alt rock": "alternative rock",
    "alt-rock": "alternative rock",
    "electronica": "electronic",
    "drum and bass": "drum n bass",
    "dnb": "drum n bass",
}

def normalise_tag(tag):
    tag = re.sub(r"\s+", " ", tag).strip().lower()
    if tag in tag_aliases:
        tag = tag_aliases[tag]
    return tag

raw_tags = tables["tags"].set_index("id")["tags"]
track_tags = {}
for track_id in clean["id"]:
    value = raw_tags.get(track_id)
    tags_here = []
    if isinstance(value, str):
        for tag in value.split(","):
            tag = normalise_tag(tag)
            if tag != "" and tag not in tags_here:
                tags_here.append(tag)
    track_tags[track_id] = tags_here

raw_count = set()
for track_id in clean["id"]:
    value = raw_tags.get(track_id)
    if isinstance(value, str):
        for tag in value.split(","):
            if tag.strip() != "":
                raw_count.add(tag.strip())

normalised_count = set()
for track_id in track_tags:
    for tag in track_tags[track_id]:
        normalised_count.add(tag)

no_tags = 0
for track_id in track_tags:
    if len(track_tags[track_id]) == 0:
        no_tags = no_tags + 1

print("Unique tags before:", len(raw_count))
print("Unique tags after:", len(normalised_count))
print("Tracks with no tags:", no_tags)

Unique tags before: 14381
Unique tags after: 14365
Tracks with no tags: 0


### Build the Exclusion List
- Listener tags that describe the person and not the music
- Tags where one artist owns more than half the songs, which leak artist identity

In [12]:
listener_tags = [
    "seen live",
    "albums i own",
    "favourites",
    "favorites",
    "my music",
    "owned",
    "want to see live",
    "favourite songs",
    "favorite songs",
    "beautiful",
    "love at first listen",
    "songs i love",
    "my favourites",
    "my favorites",
]

tag_counts = {}
tag_artists = {}
artist_of = clean.set_index("id")["artist"]
for track_id in track_tags:
    artist = artist_of[track_id]
    for tag in track_tags[track_id]:
        tag_counts[tag] = tag_counts.get(tag, 0) + 1
        if tag not in tag_artists:
            tag_artists[tag] = {}
        tag_artists[tag][artist] = tag_artists[tag].get(artist, 0) + 1

artist_dominated = []
for tag in tag_counts:
    if tag_counts[tag] < 20:
        continue
    top_artist_count = max(tag_artists[tag].values())
    if top_artist_count / tag_counts[tag] > 0.5:
        artist_dominated.append(tag)

excluded = set(listener_tags) | set(artist_dominated)

print("Listener tags excluded:", len(set(listener_tags) & set(tag_counts)))
print("Artist dominated tags excluded:", len(artist_dominated))
print("Total excluded:", len(excluded & set(tag_counts)))
print("Examples of artist dominated:", sorted(artist_dominated)[:10])

Listener tags excluded: 7
Artist dominated tags excluded: 232
Total excluded: 239
Examples of artist dominated: ['2pac', '3 doors down', 'a cappella', 'a day to remember', 'a-ha', 'abba', 'acid jazz', 'aerosmith', 'albumsiown', 'alicia keys']


### Build the Target Matrix
- Take the top N tags that survive the exclusion list
- One binary column per tag, one row per track

In [13]:
import numpy as np

kept_counts = {}
for tag in tag_counts:
    if tag not in excluded:
        kept_counts[tag] = tag_counts[tag]

ranked = sorted(kept_counts, key=lambda tag: kept_counts[tag], reverse=True)

def build_matrix(top_n):
    vocabulary = ranked[:top_n]
    position = {}
    for i in range(len(vocabulary)):
        position[vocabulary[i]] = i
    matrix = np.zeros((len(clean), len(vocabulary)), dtype=np.int8)
    row = 0
    for track_id in clean["id"]:
        for tag in track_tags[track_id]:
            if tag in position:
                matrix[row, position[tag]] = 1
        row = row + 1
    return vocabulary, matrix

for top_n in [20, 50, 100]:
    vocabulary, matrix = build_matrix(top_n)
    cardinality = matrix.sum(axis=1).mean()
    print("N:", top_n,
          "shape:", matrix.shape,
          "cardinality:", round(cardinality, 2),
          "density:", round(cardinality / top_n, 4),
          "rows with no tag:", int((matrix.sum(axis=1) == 0).sum()))

vocabulary, target = build_matrix(50)

print("Support of the smallest tag at N = 50:", int(target.sum(axis=0).min()))
print("Support of the largest tag at N = 50:", int(target.sum(axis=0).max()))

N: 20 shape: (82810, 20) cardinality: 1.74 density: 0.0871 rows with no tag: 14547


N: 50 shape: (82810, 50) cardinality: 2.46 density: 0.0493 rows with no tag: 5085


N: 100 shape: (82810, 100) cardinality: 2.96 density: 0.0296 rows with no tag: 1736


Support of the smallest tag at N = 50: 1310
Support of the largest tag at N = 50: 20727


### Save the Tags and Vocabulary

In [14]:
vocabulary_frame = pd.DataFrame({
    "tag": vocabulary,
    "support": target.sum(axis=0),
})
vocabulary_frame.to_csv(out_dir / "tag_vocabulary_50.csv", index=False)

tag_rows = []
for track_id in clean["id"]:
    keep = []
    for tag in track_tags[track_id]:
        if tag not in excluded:
            keep.append(tag)
    tag_rows.append({"id": track_id, "tags": ",".join(keep)})

tags_frame = pd.DataFrame(tag_rows)
tags_frame.to_csv(out_dir / "tags_normalised.csv", index=False)

excluded_frame = pd.DataFrame({"tag": sorted(excluded & set(tag_counts))})
excluded_frame.to_csv(out_dir / "excluded_tags.csv", index=False)

print("Wrote:", out_dir / "tag_vocabulary_50.csv")
print("Wrote:", out_dir / "tags_normalised.csv")
print("Wrote:", out_dir / "excluded_tags.csv")
vocabulary_frame.head()

Wrote: data\processed\tag_vocabulary_50.csv
Wrote: data\processed\tags_normalised.csv
Wrote: data\processed\excluded_tags.csv


,tag,support
0,rock,20727
1,pop,16072
2,indie,14155
3,alternative,11126
4,female vocalists,9398


### Build the Random Split
- Seeded 70/15/15 over the tracks that survived cleaning

In [15]:
track_ids = list(clean["id"])

rng = np.random.default_rng(seed)
shuffled = list(track_ids)
rng.shuffle(shuffled)

n_total = len(shuffled)
n_train = int(0.70 * n_total)
n_valid = int(0.15 * n_total)

random_split = {}
for track_id in shuffled[:n_train]:
    random_split[track_id] = "train"
for track_id in shuffled[n_train:n_train + n_valid]:
    random_split[track_id] = "valid"
for track_id in shuffled[n_train + n_valid:]:
    random_split[track_id] = "test"

counts = {"train": 0, "valid": 0, "test": 0}
for track_id in random_split:
    counts[random_split[track_id]] = counts[random_split[track_id]] + 1

print("Total tracks:", n_total)
print("Random split:", counts)

Total tracks: 82810
Random split: {'train': 57966, 'valid': 12421, 'test': 12423}


### Build the Artist Disjoint Split
- Whole artists go to one part, so no artist appears in two parts
- Artists are shuffled then filled until each part reaches its share

In [16]:
artist_tracks = {}
for track_id in track_ids:
    artist = artist_of[track_id]
    if artist not in artist_tracks:
        artist_tracks[artist] = []
    artist_tracks[artist].append(track_id)

artists = list(artist_tracks)
rng = np.random.default_rng(seed)
rng.shuffle(artists)

artist_split = {}
filled = 0
for artist in artists:
    if filled < n_train:
        part = "train"
    elif filled < n_train + n_valid:
        part = "valid"
    else:
        part = "test"
    for track_id in artist_tracks[artist]:
        artist_split[track_id] = part
    filled = filled + len(artist_tracks[artist])

counts = {"train": 0, "valid": 0, "test": 0}
for track_id in artist_split:
    counts[artist_split[track_id]] = counts[artist_split[track_id]] + 1

parts_per_artist = {}
for track_id in artist_split:
    artist = artist_of[track_id]
    if artist not in parts_per_artist:
        parts_per_artist[artist] = set()
    parts_per_artist[artist].add(artist_split[track_id])

shared_artists = 0
for artist in parts_per_artist:
    if len(parts_per_artist[artist]) > 1:
        shared_artists = shared_artists + 1

print("Artists:", len(artists))
print("Artist disjoint split:", counts)
print("Artists appearing in more than one part:", shared_artists)

Artists: 11803
Artist disjoint split: {'train': 57997, 'valid': 12424, 'test': 12389}
Artists appearing in more than one part: 0


### Pick the Shared Vocabulary
- Support is counted only on tracks that are training under both splits
- Neither split can see its own held out labels when the vocabulary is chosen

In [17]:
shared_train = []
for track_id in track_ids:
    if random_split[track_id] == "train" and artist_split[track_id] == "train":
        shared_train.append(track_id)

shared_counts = {}
for track_id in shared_train:
    for tag in track_tags[track_id]:
        if tag not in excluded:
            shared_counts[tag] = shared_counts.get(tag, 0) + 1

shared_ranked = sorted(shared_counts, key=lambda tag: shared_counts[tag], reverse=True)
shared_vocabulary = shared_ranked[:50]

print("Tracks training under both splits:", len(shared_train))
print("Tags available there:", len(shared_counts))
print("Smallest support in the top 50:", shared_counts[shared_vocabulary[49]])
print("Largest support in the top 50:", shared_counts[shared_vocabulary[0]])
print("Overlap with the earlier top 50:", len(set(shared_vocabulary) & set(vocabulary)))

Tracks training under both splits: 40537
Tags available there: 8799
Smallest support in the top 50: 641
Largest support in the top 50: 9818
Overlap with the earlier top 50: 49


### Report Per Tag Counts
- Positives for every tag in each part of both splits
- Tags with no test positives cannot be scored, so they are flagged

In [18]:
def count_positives(split_map, vocab):
    rows = []
    for tag in vocab:
        row = {"tag": tag, "train": 0, "valid": 0, "test": 0}
        rows.append(row)
    by_tag = {}
    for row in rows:
        by_tag[row["tag"]] = row
    wanted = set(vocab)
    for track_id in track_ids:
        part = split_map[track_id]
        for tag in track_tags[track_id]:
            if tag in wanted:
                by_tag[tag][part] = by_tag[tag][part] + 1
    return pd.DataFrame(rows)

random_counts = count_positives(random_split, shared_vocabulary)
artist_counts = count_positives(artist_split, shared_vocabulary)

per_tag = random_counts.merge(artist_counts, on="tag", suffixes=("_random", "_artist"))

print("Tags with no test positives, random split:", int((per_tag["test_random"] == 0).sum()))
print("Tags with no test positives, artist split:", int((per_tag["test_artist"] == 0).sum()))
print("Smallest test support, random split:", int(per_tag["test_random"].min()))
print("Smallest test support, artist split:", int(per_tag["test_artist"].min()))

per_tag.sort_values("test_random").head(10)

Tags with no test positives, random split: 0
Tags with no test positives, artist split: 0
Smallest test support, random split: 169
Smallest test support, artist split: 107


,tag,train_random,valid_random,test_random,train_artist,valid_artist,test_artist
47,funk,837,169,169,902,166,107
48,shoegaze,947,189,174,925,150,235
46,fip,928,189,193,916,194,200
41,jazz,1026,194,197,1041,207,169
44,pop rock,940,200,211,1005,236,110
39,country,1129,213,212,1074,232,248
42,cover,1011,223,214,1015,206,227
40,dream pop,1110,226,216,1060,271,221
35,60s,1157,272,238,1232,264,171
45,lo-fi,1075,235,243,1022,230,301


### Save the Splits and Per Tag Counts

In [19]:
split_rows = []
for track_id in track_ids:
    split_rows.append({
        "id": track_id,
        "artist": artist_of[track_id],
        "random_split": random_split[track_id],
        "artist_split": artist_split[track_id],
    })

splits_frame = pd.DataFrame(split_rows)
splits_frame.to_csv(out_dir / "splits.csv", index=False)

shared_frame = pd.DataFrame({
    "tag": shared_vocabulary,
    "support": [shared_counts[tag] for tag in shared_vocabulary],
})
shared_frame.to_csv(out_dir / "shared_vocabulary_50.csv", index=False)

per_tag.to_csv(out_dir / "per_tag_counts.csv", index=False)

print("Wrote:", out_dir / "splits.csv")
print("Wrote:", out_dir / "shared_vocabulary_50.csv")
print("Wrote:", out_dir / "per_tag_counts.csv")
per_tag.head()

Wrote: data\processed\splits.csv
Wrote: data\processed\shared_vocabulary_50.csv
Wrote: data\processed\per_tag_counts.csv


,tag,train_random,valid_random,test_random,train_artist,valid_artist,test_artist
0,rock,14426,3161,3140,14096,3521,3110
1,pop,11256,2362,2454,11875,2195,2002
2,indie,9979,2159,2017,9961,2113,2081
3,alternative,7823,1636,1667,7852,1527,1747
4,female vocalists,6628,1394,1376,6818,1252,1328


### Compare Artist Coverage
- Under the random split most test artists also appear in training
- Under the artist disjoint split none of them do, which is the point of it

In [20]:
def artist_coverage(split_map):
    train_artists = set()
    test_artists = set()
    for track_id in track_ids:
        artist = artist_of[track_id]
        if split_map[track_id] == "train":
            train_artists.add(artist)
        if split_map[track_id] == "test":
            test_artists.add(artist)
    seen = test_artists & train_artists
    tracks_by_seen = 0
    for track_id in track_ids:
        if split_map[track_id] == "test" and artist_of[track_id] in seen:
            tracks_by_seen = tracks_by_seen + 1
    return train_artists, test_artists, seen, tracks_by_seen

for name in ["random", "artist"]:
    if name == "random":
        split_map = random_split
    else:
        split_map = artist_split
    train_artists, test_artists, seen, tracks_by_seen = artist_coverage(split_map)
    print(name, "split")
    print("  train artists:", len(train_artists))
    print("  test artists:", len(test_artists))
    print("  test artists also in train:", len(seen))
    print("  test tracks by an artist seen in train:", tracks_by_seen)

random split
  train artists: 10150
  test artists: 4794
  test artists also in train: 3938
  test tracks by an artist seen in train: 11511


artist split
  train artists: 8297
  test artists: 1723
  test artists also in train: 0
  test tracks by an artist seen in train: 0


### Compare Lost Test Positives
- A tag is lost when it has training positives but none in test
- Checked at the three vocabulary sizes the project scores on

In [21]:
def lost_tags(split_map, vocab):
    wanted = set(vocab)
    train_positives = {}
    test_positives = {}
    for tag in vocab:
        train_positives[tag] = 0
        test_positives[tag] = 0
    for track_id in track_ids:
        part = split_map[track_id]
        for tag in track_tags[track_id]:
            if tag in wanted:
                if part == "train":
                    train_positives[tag] = train_positives[tag] + 1
                elif part == "test":
                    test_positives[tag] = test_positives[tag] + 1
    lost = []
    for tag in vocab:
        if train_positives[tag] > 0 and test_positives[tag] == 0:
            lost.append(tag)
    return lost

for top_n in [20, 50, 100]:
    vocab = shared_ranked[:top_n]
    random_lost_n = lost_tags(random_split, vocab)
    artist_lost_n = lost_tags(artist_split, vocab)
    print("N:", top_n,
          "lost random:", len(random_lost_n),
          "lost artist:", len(artist_lost_n),
          "smallest support:", shared_counts[vocab[top_n - 1]])

random_lost = lost_tags(random_split, shared_vocabulary)
artist_lost = lost_tags(artist_split, shared_vocabulary)

print("Lost in the random split at N = 50:", len(random_lost))
print("Lost in the artist disjoint split at N = 50:", len(artist_lost))

N: 20 lost random: 0 lost artist: 0 smallest support: 1509


N: 50 lost random: 0 lost artist: 0 smallest support: 641


N: 100 lost random: 0 lost artist: 0 smallest support: 238


Lost in the random split at N = 50: 0
Lost in the artist disjoint split at N = 50: 0


In [22]:
rows = []
for tag in shared_vocabulary:
    train_random = 0
    test_random = 0
    train_artist = 0
    test_artist = 0
    for track_id in track_ids:
        if tag in track_tags[track_id]:
            if random_split[track_id] == "train":
                train_random = train_random + 1
            elif random_split[track_id] == "test":
                test_random = test_random + 1
            if artist_split[track_id] == "train":
                train_artist = train_artist + 1
            elif artist_split[track_id] == "test":
                test_artist = test_artist + 1
    rows.append({
        "tag": tag,
        "support": shared_counts[tag],
        "test_random": test_random,
        "test_artist": test_artist,
        "ratio": round(test_artist / max(test_random, 1), 2),
    })

tag_frame = pd.DataFrame(rows).sort_values("ratio")

print("Tags where the artist split leaves the fewest test positives:")
print(tag_frame.head(8))

Tags where the artist split leaves the fewest test positives:
           tag  support  test_random  test_artist  ratio
20   hard rock     1497          549          261   0.48
44    pop rock      711          211          110   0.52
47        funk      654          169          107   0.63
15       dance     1961          590          421   0.71
35         60s      855          238          171   0.72
5   electronic     4192         1240          937   0.76
12         80s     2289          700          537   0.77
26    new wave     1207          357          284   0.80


### Save the Comparison

In [23]:
coverage_rows = []
for name in ["random", "artist"]:
    if name == "random":
        split_map = random_split
        lost = random_lost
    else:
        split_map = artist_split
        lost = artist_lost
    train_artists, test_artists, seen, tracks_by_seen = artist_coverage(split_map)
    coverage_rows.append({
        "split": name,
        "train_artists": len(train_artists),
        "test_artists": len(test_artists),
        "test_artists_seen_in_train": len(seen),
        "test_tracks_by_seen_artist": tracks_by_seen,
        "lost_tags": len(lost),
    })

coverage_frame = pd.DataFrame(coverage_rows)
coverage_frame.to_csv(out_dir / "split_comparison.csv", index=False)
tag_frame.to_csv(out_dir / "test_positives_by_tag.csv", index=False)

print("Wrote:", out_dir / "split_comparison.csv")
print("Wrote:", out_dir / "test_positives_by_tag.csv")
coverage_frame

Wrote: data\processed\split_comparison.csv
Wrote: data\processed\test_positives_by_tag.csv


,split,train_artists,test_artists,test_artists_seen_in_train,test_tracks_by_seen_artist,lost_tags
0,random,10150,4794,3938,11511,0
1,artist,8297,1723,0,0,0
